## DATA EXTRACTION (text, tables, images, diagrams, charts, graphs, and OCR for PDF-only input.)

In [87]:

import os 
from pathlib import Path
import json
from typing import List, Dict, Tuple, Any
from tqdm import tqdm

#document parsing & pdf
from docling.document_converter import DocumentConverter
import fitz
import pdfplumber
from pdf2image import convert_from_path

#tables
import camelot
import pandas as pd
import uuid

#ocr and images
import pytesseract
from PIL import Image

#cv
import cv2
import numpy as np

#helpers
import magic
import shutil 



In [88]:
#paths
RAW_DIR = Path("../data/raw")
EXTRACTED_DIR = Path("../data/extracted")
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

for sub in ["json", "tables", "images", "ocr", "metadata"]:
    (EXTRACTED_DIR / sub).mkdir(parents=True, exist_ok=True)

print("extracted:", EXTRACTED_DIR)

extracted: ../data/extracted


In [89]:
# helpers

def is_pdf(path: Path) -> bool:
    try:
        t = magic.from_file(str(path), mime=True)
        return t == "application/pdf"
    except Exception:
        return False

def save_json(data: dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def safe_mkdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def normalize_bbox(bbox: Tuple[int,int,int,int], page_width:int, page_height:int):
    x0,y0,x1,y1 = map(int, bbox)
    x0 = max(0, min(x0, page_width))
    x1 = max(0, min(x1, page_width))
    y0 = max(0, min(y0, page_height))
    y1 = max(0, min(y1, page_height))
    return [x0,y0,x1,y1]



In [90]:
#Initialize Docling & basic PDF meta
converter = DocumentConverter()

def extract_pdf_metadata(pdf_path: Path)->dict:
    try:
        doc = fitz.open(pdf_path)
        meta = doc.metadata
        info = {
            "file_name": pdf_path.name,
            "path": str(pdf_path),
            "pages": len(doc),
            "title": meta.get("title"),
            "author": meta.get("author"),
            "filesize_kb": round(pdf_path.stat().st_size/ 1024, 2)
        }
        doc.close()
        return info
    except Exception as e:
        return {"file_name":pdf_path.name, "error":str(e)}

def save_metadata(metadata: Dict):
    pdf_name = "sample"
    output_folder = EXTRACTED_DIR/"metadata"/f"{pdf_name}_metadata.json"
    try:
        with open(output_folder, 'w', encoding='utf-8') as f:
            json.dump(metadata,f,ensure_ascii=False,indent=4)
        return output_folder
    except IOError as e:
        print(f"error saving the file {output_folder}:{e}")
        return None

In [91]:
# metadata = extract_pdf_metadata(RAW_DIR/"sample.pdf")
# save_metadata(metadata)

In [109]:
# text extractor (dockling + pymupdf)

def extract_text_docling(pdf_path: Path) -> dict:
   result = {
      "docling_json": None,
      "page_texts": [],
      "extractor": None
   }
   try:
      converter = DocumentConverter()
      res = converter.convert(str(pdf_path))
      doc = res.document

      json_path = EXTRACTED_DIR / "json" / f"{pdf_path.stem}_docling.json"
      save_json(doc.export_to_dict(), json_path)
      result["docling_json"] = str(json_path)

      try:
         doc_fitz = fitz.open(str(pdf_path))
         for page in doc_fitz:
            result["page_texts"].append(page.get_text("text")) 
         doc_fitz.close()
      except:
         full_text = (
            doc.export_to_text()
            if hasattr(doc, "export_to_text")
            else doc.export_to_markdown()
         )
         result["page_texts"] = [full_text]

      result["extractor"] = "docling + pymupdf"
      return result
   
   except:
      try:
         result["extractor"] = "pymupdf"
         doc_fitz = fitz.open(str(pdf_path))
         for page in doc_fitz:
            result["page_texts"].append(page.get_text("text")) 
         doc_fitz.close()

      except Exception as e2:
         result["page_texts"] = []
         result["error"] = str(e2)

      return result



    

In [93]:
# extract_text_docling(RAW_DIR/"sample.pdf")

In [110]:
def extract_tables(pdf_path: Path) -> List[Dict[str, Any]]:
    tables = []
    try:
        tables_lattice = camelot.read_pdf(
            str(pdf_path),
            pages="all",
            flavor="lattice",
            edge_tol=50
        )
    except Exception:
        tables_lattice = []

    #save lattice tables
    if tables_lattice:
        for i, t in enumerate(tables_lattice):
            csv_path = EXTRACTED_DIR / "tables" / f"{pdf_path.stem}_lattice_{i}.csv"
            t.to_csv(str(csv_path), index = False)
            tables.append({
                "csv_path": str(csv_path),
                "pages": None,
                "extractor": "lattice"
            })
    try:
        tables_stream = camelot.read_pdf(
            str(pdf_path),
            pages="all",
            flavor="stream"
        )
    except Exception:
        tables_stream = []

    #save stream tables
    if tables_stream:
        for i, t in enumerate(tables_stream):
            csv_path = EXTRACTED_DIR / "tables" / f"{pdf_path.stem}_stream_{i}.csv"
            t.to_csv(str(csv_path), index = False)
            tables.append({
                "csv_path": str(csv_path),
                "pages": None,
                "extractor": "stream"
            })

    return tables


In [95]:
# extract_tables(RAW_DIR/"sample.pdf")

In [96]:
# [
#   {
#     "csv_path": "../data/extracted_industrial/tables/report_lattice_0.csv",
#     "page": null,
#     "extractor": "lattice"
#   },
#   {
#     "csv_path": "../data/extracted_industrial/tables/report_lattice_1.csv",
#     "page": null,
#     "extractor": "lattice"
#   },
#   {
#     "csv_path": "../data/extracted_industrial/tables/report_stream_0.csv",
#     "page": null,
#     "extractor": "stream"
#   }
# ]


In [97]:
# img = (
#     15,          # img[0] → xref
#     0,           # img[1]
#     400,         # img[2] → width
#     300,         # img[3] → height
#     8,           # img[4]
#     'DeviceRGB', # img[5]
#     '',          # img[6]
#     'Im1'        # img[7]
# )

# (xref, smask, width, height, bpc, colorspace, alt, name, ...)
# xref = cross-reference number
# xref = img[0]


# A PDF is built like a database

# Every object (text, image, font) has a unique ID

# That ID is called xref

# 📌 Think of it like:

# Concept	Real-world analogy
# xref	Primary key in DB
# PDF object	Row in DB
# Image data	Stored separately

In [111]:
def extract_images_with_bbox(pdf_path: Path) -> List[Dict[str, Any]]:
    output = []

    #PyMuPDF extraction (embedded images)
    doc = fitz.open(str(pdf_path))
    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        images = page.get_images(full=True)

        for img_idx, img in enumerate(images):
            #xref unique no for each image
            xref = img[0]
            #extracting binaries in image
            try:
                base = doc.extract_image(xref)
                # base = {
                # "image": b"\x89PNG\r\n\x1a\n\x00\x00...",
                # "width": 512,
                # "height": 256,
                # "ext": "png",
                # "colorspace": "DeviceRGB"
                # }
            except Exception:
                continue
            img_bytes = base["image"]
            ext = base.get("ext","png")

            img_name = f"{pdf_path.stem}_p{page_index+1}_embed_{img_idx}.{ext}"
            img_path = EXTRACTED_DIR / "images" / img_name
            with open(img_path, "wb") as f:
                f.write(img_bytes)

            bbox = None
            try:
                for inst in page.get_images(full=True):
                    if inst[0] == xref and len(inst) >= 9:
                        # Some PyMuPDF versions include a rect at index 7 or later
                        rect = None
                        if len(inst) > 7:
                            value = inst[7]
                            if isinstance(value, fitz.Rect):
                                rect  = value
                        if rect:
                            bbox = [rect.x0, rect.y0, rect.x1, rect.y1]
                        break
            except:
                bbox = None

            output.append(
                {
                    "img_path": str(img_path),
                    "page": page_index + 1,
                    "bbox": bbox,
                    "width": base.get("width"),
                    "height": base.get("height"),
                    "extractor": "pymupdf"
                }
            )
    doc.close()

    #pdf2image extraction (page render images)
    #charts, histograms, pie charts are NOT embedded images, only vector drawings

    pages = convert_from_path(str(pdf_path), dpi=200)
    for i , pil_img in enumerate(pages):
        img_name = f"{pdf_path.stem}_p{i+1}_render_{uuid.uuid4().hex}.png"
        img_path = EXTRACTED_DIR / "images" / img_name
        pil_img.save(img_path, "PNG")

        output.append(
            {
                "img_path": str(img_path),
                "page": i + 1,
                "bbox": None,
                "width": pil_img.width,
                "height": pil_img.height,
                "extractor": "pdf2image"  
            }
        )
    
    return output

In [99]:
# extract_images_with_bbox(RAW_DIR / "sample.pdf")

In [112]:
def extract_ocr(img_path : Path):
    img_dir = EXTRACTED_DIR / "images"
    all_images = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    ocr_path = EXTRACTED_DIR / "ocr"
    for i,image in enumerate(all_images):
        print(type(image))
        text = pytesseract.image_to_string(str(image))
        file_name = f"ocr{i}.txt"
        file_path = ocr_path / file_name
        with open(file_path, "w",encoding="utf-8") as f:
            f.write(text)



In [101]:
# extract_ocr(RAW_DIR)

In [102]:
# RAW_DIR = Path("../data/raw")
# image_path = EXTRACTED_DIR /  "images"  
# def into_image(pdf_path : Path):
#     images = convert_from_path(pdf_path)

#     for i, image in enumerate(images):
#         #pages = [PIL.image.image, PIL.image.image,....]
#         image.save(image_path/f"{i}.jpeg", "JPEG")
#     return

# into_image(Path("../data/raw/sample.pdf"))

In [113]:
def process_pdf(pdf_path: Path):

    pipeline_status: Dict[str, Any] = {"file_name" : pdf_path.name}
    try:
        print("attempting metadata extraction...")
        #extract metadata
        metadata = extract_pdf_metadata(pdf_path)
        save_metadata(metadata)
        pipeline_status["metadata_status"] = "success"
    except Exception as e:
        print(f"error in attempting metadata extraction {e}")
        pipeline_status["metadata_status"] = f"failed:{str(e)}"

    try:
        #extract json
        print("attempting text  extraction...")
        extract_text_docling(pdf_path)
        pipeline_status["text_status"] = "success"
    except Exception as e:
        print(f"error in attempting text extraction {e}")
        pipeline_status["text_status"] = f"failed:{str(e)}"

    try:
        #extract image
        print("attempting image extraction...")
        extract_images_with_bbox(pdf_path)
        pipeline_status["image_status"] = "success"
    except Exception as e:
        print(f"error in attempting image extraction {e}")
        pipeline_status["image_status"] = f"failed:{str(e)}"

    try:
        #extract tables
        print("attempting table extraction...")
        extract_tables(pdf_path)
        pipeline_status["table_status"] = "success"
    except Exception as e:
        print(f"error in attempting table extraction {e}")
        pipeline_status["table_status"] = f"failed:{str(e)}"

    try:
        #extract ocr
        print("attempting ocr extraction...")
        extract_ocr(EXTRACTED_DIR/"images")
        pipeline_status["ocr_status"] = "success"
    except Exception as e:
        print(f"error in attempting ocr extraction {e}")
        pipeline_status["ocr_status"] = f"failed:{str(e)}"

    return pipeline_status

    


In [ ]:
# extract_ocr(EXTRACTED_DIR/"images")

<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._loc

In [114]:
process_pdf(RAW_DIR/"sample.pdf")

2025-12-14 22:13:28,952 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-14 22:13:28,977 - INFO - Going to convert document batch...
2025-12-14 22:13:28,979 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-12-14 22:13:28,980 - INFO - Auto OCR model selected ocrmac.
2025-12-14 22:13:28,980 - INFO - Accelerator device: 'mps'


attempting metadata extraction...
attempting text  extraction...


2025-12-14 22:13:41,157 - INFO - Accelerator device: 'mps'
2025-12-14 22:13:42,364 - INFO - Processing document sample.pdf
2025-12-14 22:14:12,726 - INFO - Finished converting document sample.pdf in 43.78 sec.


attempting image extraction...
attempting table extraction...


/Users/sha/Developer/contextIQ/.venv/lib/python3.13/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (170.927, 418.7249566, 400.9952525, 638.7759894999999)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/sha/Developer/contextIQ/.venv/lib/python3.13/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (26.134999999999998, 26.4244986, 411.58575249999996, 411.2337837999999)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/sha/Developer/contextIQ/.venv/lib/python3.13/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (26.134999999999998, 26.4244986, 411.4957525, 452.1351146470588)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/sha/Developer/contextIQ/.venv/lib/python3.13/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (15.758, 342.3063986, 401.1370525,

attempting ocr extraction...
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>
<class 'pathlib._local.Pos

{'file_name': 'sample.pdf',
 'metadata_status': 'success',
 'text_status': 'success',
 'image_status': 'success',
 'table_status': 'success',
 'ocr_status': 'success'}